In [11]:
import stable_retro as retro
from gymnasium import Env
from gymnasium.spaces import MultiBinary, Box
import numpy as np
import cv2
from matplotlib import pyplot as plt
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3 import PPO

In [12]:
class StreetFighter(Env):
    def __init__(self):
        super().__init__()
        self.observation_space = Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)
        
        self.action_space = MultiBinary(12)

        self.game = retro.make(game="StreetFighterIISpecialChampionEdition-Genesis-v0", use_restricted_actions=retro.Actions.FILTERED, render_mode=None)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.game.step(action)
        obs = self.preprocess(obs)

        self.previous_frame = obs

        if info["health"] == 0 and info["enemy_health"] == 0:
            reward = 0
            self.enemy_health = 0
            self.player_health = 0
        else:
            dmg_dealt = max(0, self.enemy_health -info["enemy_health"])
            dmg_taken = max(0, self.player_health- info["health"])
            self.enemy_health = info["enemy_health"]
            self.player_health = info["health"]
            reward = dmg_dealt - dmg_taken
        

        return obs, reward, terminated, truncated, info


    def render(self, *args, **kwargs):
        self.game.render()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs, info = self.game.reset(seed=seed, options=options)
        obs = self.preprocess(obs)
        self.previous_frame = obs

        
        info = self.game.data.lookup_all()
        self.player_health = info.get("health", 0)
        self.enemy_health = info.get("enemy_health", 0)
        return obs, info

    def preprocess(self, observation):
        gray = cv2.cvtColor(observation, cv2.COLOR_RGB2GRAY)

        resize = cv2.resize(gray, (84,84), interpolation=cv2.INTER_AREA)

        channels = np.reshape(resize, (84,84,1))
        return channels


    def close(self):
        self.game.close()

In [13]:
LOG_DIR = "./opt_logs/"
model_path = "./opt/trial_125_best_model.zip"
save_path = "./opt/sf_5m_v3"
callback = CheckpointCallback(
    save_freq=25000,
    save_path="./opt/checkpoints/",
    name_prefix="sf_5m_v3"
)
mil_timesteps = 20
timesteps = 1000000 * mil_timesteps

def make_env():
    return Monitor(StreetFighter(), LOG_DIR)

In [14]:
def main():
    env = SubprocVecEnv([make_env for _ in range(4)])
    env = VecFrameStack(env, n_stack=4, channels_order="last")
    model = PPO.load(model_path, env, device="mps", verbose=1, tensorboard_log=LOG_DIR, ent_coef=0.005)
    model.learn(total_timesteps=timesteps, progress_bar=True, callback=callback)
    model.save(save_path)
    env.close()
    print("done")

    


if __name__ == "__main__":
    main()

Logging to ./opt_logs/PPO_7


/Users/elvincheung/PersonalProjects/sf_rl/.venv-2/lib/python3.14/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

------------------------------
| time/              |       |
|    fps             | 1236  |
|    iterations      | 1     |
|    time_elapsed    | 23    |
|    total_timesteps | 28928 |
------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 8.5e+03     |
|    ep_rew_mean          | -72.8       |
| time/                   |             |
|    fps                  | 511         |
|    iterations           | 2           |
|    time_elapsed         | 113         |
|    total_timesteps      | 57856       |
| train/                  |             |
|    approx_kl            | 0.018272767 |
|    clip_fraction        | 0.0367      |
|    clip_range           | 0.377       |
|    entropy_loss         | -8.27       |
|    explained_variance   | -0.0434     |
|    learning_rate        | 6.26e-05    |
|    loss                 | 2.87        |
|    n_updates            | 12          |
|    policy_gradient_loss |

KeyboardInterrupt: 